<a href="https://colab.research.google.com/github/JM-NGUMI/AI-course-work-for-science-and-engineering/blob/main/Joseph_Assigment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment Five:Building My First Supervised Machine Learning Model

# Targets ;concentrations of pollutants C(x)i, j, k, l,at fall outdistance x
Features; Concentration of Qi,j,k,l, in g/s, Stack height (H) in meters, Wind speed (u) in m/s and constants σ_y⋅σ_z depending on stability class of the data
Where: I,j,k,l,m are PM2.5, PM10 CO, NO2 pollutants respectively



# My problem is a regression problem because Predicting continuous numerical quantities as a target is the defining characteristic of regression.
A Classification algorithms predict discrete class labels or categorical buckets (such as "Low / Medium / High" or "Unsafe / Safe") but my goal is to predict precise numerical concentration levels based on physical input features (Q, H, u, fallout distance x, and atmospheric stability parameters), a regression algorithm (e.g., Gaussian Plume Model modeling via Multiple Linear Regression, Random Forest Regressor, or Neural Networks) is require


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the dataset from the Excel file
file_path = '/content/AI FOR SCIENCE AND ENGINEERING PROJECT DATA.xlsx'
df = pd.read_excel(file_path, sheet_name='SITE DATA')

# 2. Select Features (X) and Targets (y)
# Meteorological features: Temperature, Relative Humidity, Wind Speed, Wind Direction
features = ['TEMPERATURE oC', 'RH (%)', 'wind speed v, m/s', 'wind direction ,degrees']

# Target pollutant concentrations (or select a single target like 'PM2.5(ug/m3)')
targets = ['PM2.5(ug/m3)', 'PM10(ug/m3)', 'NO2 , ppm', ' CO , ppm']

# Clean dataset by removing any rows with missing values in key columns
df_clean = df.dropna(subset=features + targets)

X = df_clean[features]
y = df_clean[targets]  # Or y = df_clean['PM2.5(ug/m3)'] for single-target regression

# 3. Perform 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 4. Display the shapes of resulting split datasets
print(f"Total dataset size: {len(df_clean)} rows")
print(f"Training features shape (X_train): {X_train.shape}")  # (288, 4)
print(f"Testing features shape  (X_test) : {X_test.shape}")   # (73, 4)
print(f"Training targets shape  (y_train): {y_train.shape}")  # (288, 4)
print(f"Testing targets shape   (y_test) : {y_test.shape}")   # (73, 4)

Total dataset size: 361 rows
Training features shape (X_train): (288, 4)
Testing features shape  (X_test) : (73, 4)
Training targets shape  (y_train): (288, 4)
Testing targets shape   (y_test) : (73, 4)


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load dataset
file_path = '/content/AI FOR SCIENCE AND ENGINEERING PROJECT DATA.xlsx'
df = pd.read_excel(file_path, sheet_name='SITE DATA')

# 2. Define Features and Target
features = ['TEMPERATURE oC', 'RH (%)', 'wind speed v, m/s', 'wind direction ,degrees']
target = 'PM2.5(ug/m3)'

df_clean = df.dropna(subset=features + [target])
X = df_clean[features]
y = df_clean[target]

# 3. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 4. Model 1: Multiple Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# 5. Model 2: Decision Tree Regressor
dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

# 6. Evaluation Function
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# Evaluate both models
results = [
    evaluate_model(y_test, y_pred_lr, 'Linear Regression'),
    evaluate_model(y_test, y_pred_dt, 'Decision Tree Regressor')
]

# Display comparative results
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                  Model      MAE       MSE     RMSE       R2
      Linear Regression 2.621383 14.724597 3.837264 0.388338
Decision Tree Regressor 1.553527  6.811357 2.609858 0.717055


Model Performance Comparison:
Evaluating both models on the 20% holdout test set (73 samples) yields the following metrics for predicting $\text{PM}_{2.5}$ concentration:MetricLinear RegressionDecision Tree Regressor (max_depth=5)Best PerformerMean Absolute Error (MAE)2.62141.5535Decision Tree (Lower error)Mean Squared Error (MSE)14.72466.8114Decision Tree (Lower error)Root Mean Squared Error (RMSE)3.83732.6099Decision Tree (Lower error)Coefficient of Determination ($R^2$)0.3883 (38.8%)0.7171 (71.7%)Decision Tree (Higher accuracy)
Key Differences Between the Two Models Linearity vs. Non-Linearity:
Linear Regression assumes a strict linear relationship between meteorological variables (wind speed, temperature, humidity) and pollutant concentration. Because real atmospheric dispersion is non-linear, Linear Regression achieves an $R^2$ of only 38.8%.Decision Tree Regressor splits the feature space into non-linear decision regions, capturing complex interactions between weather parameters and achieving a significantly higher $R^2$ of 71.7%.Error Magnitudes:On average, the Decision Tree predictions deviate by 1.55 $\mu\text{g/m}^3$ (MAE) compared to 2.62 $\mu\text{g/m}^3$ for Linear Regression.Interpretability & Extrapolation:
Linear Regression provides global coefficients for each variable (e.g., effect per unit change in wind speed) and can extrapolate outside the training data range.Decision Tree creates step-function decision rules (e.g., if temperature > 16.5 and RH <= 62%), making it strong for local patterns but unable to extrapolate beyond the maximum/minimum target values seen in training